In [35]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

INPUT_DIR = Path('input')  # <-- đổi đường dẫn nếu cần

# Màu sắc nhất quán
COLOR_MISSING  = '#E24B4A'
COLOR_OK       = '#1D9E75'
COLOR_WARN     = '#EF9F27'
COLOR_NEUTRAL  = '#888780'

print('✅ Imports OK')

✅ Imports OK


In [36]:
FILE_CONFIGS = {
    'products'       : {'pk': 'product_id',  'date_cols': []},
    'customers'      : {'pk': 'customer_id', 'date_cols': ['signup_date']},
    'promotions'     : {'pk': 'promo_id',    'date_cols': ['start_date', 'end_date']},
    'geography'      : {'pk': 'zip',         'date_cols': []},
    'orders'         : {'pk': 'order_id',    'date_cols': ['order_date']},
    'order_items'    : {'pk': None,          'date_cols': []},
    'payments'       : {'pk': 'order_id',    'date_cols': []},
    'shipments'      : {'pk': 'order_id',    'date_cols': ['ship_date', 'delivery_date']},
    'returns'        : {'pk': 'return_id',   'date_cols': ['return_date']},
    'reviews'        : {'pk': 'review_id',   'date_cols': ['review_date']},
    'inventory'      : {'pk': None,          'date_cols': ['snapshot_date']},
    'web_traffic'    : {'pk': None,          'date_cols': ['date']},
    'sales'          : {'pk': None,          'date_cols': ['Date']},
    'sample_submission': {'pk': None,        'date_cols': ['Date']},
}

dfs = {}
load_errors = []

for name, cfg in FILE_CONFIGS.items():
    path = INPUT_DIR / f'{name}.csv'
    try:
        df = pd.read_csv(
            path,
            parse_dates=cfg['date_cols'] if cfg['date_cols'] else False,
            low_memory=False
        )
        dfs[name] = df
        print(f'  ✅ {name:<22} {df.shape[0]:>8,} rows  x  {df.shape[1]:>3} cols')
    except FileNotFoundError:
        load_errors.append(name)
        print(f'  ❌ {name} — FILE NOT FOUND tại {path}')
    except Exception as e:
        load_errors.append(name)
        print(f'  ⚠️  {name} — LỖI: {e}')

print(f'\nLoaded {len(dfs)}/14 files.')

  ✅ products                  2,412 rows  x    8 cols
  ✅ customers               121,930 rows  x    7 cols
  ✅ promotions                   50 rows  x   10 cols
  ✅ geography                39,948 rows  x    4 cols
  ✅ orders                  646,945 rows  x    8 cols
  ✅ order_items             714,669 rows  x    7 cols
  ✅ payments                646,945 rows  x    4 cols
  ✅ shipments               566,067 rows  x    4 cols
  ✅ returns                  39,939 rows  x    7 cols
  ✅ reviews                 113,551 rows  x    7 cols
  ✅ inventory                60,247 rows  x   17 cols
  ✅ web_traffic               3,652 rows  x    7 cols
  ✅ sales                     3,833 rows  x    3 cols
  ✅ sample_submission           548 rows  x    3 cols

Loaded 14/14 files.


In [37]:
# ============================================================
# MASTER DATA — Business Logic Validation
# Paste các cell này vào sau notebook 00_data_quality_check
# ============================================================

# ── CELL 1: products — Product Versioning Analysis ──────────
print("=" * 65)
print("📦 PRODUCTS — Kiểm tra Product Versioning")
print("=" * 65)

prod = dfs['products'].copy()

# Số product_id duy nhất vs số tổ hợp (name, size, color) duy nhất
n_ids        = prod['product_id'].nunique()
n_identity   = prod[['product_name','size','color']].drop_duplicates().shape[0]
n_duplicated = prod.duplicated(subset=['product_name','size','color'], keep=False).sum()

print(f"  Tổng product_id       : {n_ids:,}")
print(f"  Tổng (name,size,color): {n_identity:,}")
print(f"  Rows bị lặp identity  : {n_duplicated:,}  ({n_duplicated/n_ids*100:.1f}%)")
print()

# Top 10 mẫu hàng bị versioning nhiều nhất
dup_groups = (
    prod.groupby(['product_name','size','color'])
    .agg(n_versions=('product_id','count'),
         price_min=('price','min'),
         price_max=('price','max'),
         cogs_min=('cogs','min'),
         cogs_max=('cogs','max'))
    .query('n_versions > 1')
    .sort_values('n_versions', ascending=False)
)
print(f"  Số SKU có >1 version  : {len(dup_groups):,}")
print()
print("  Top 10 SKU nhiều version nhất:")
print(dup_groups.head(10).to_string())

📦 PRODUCTS — Kiểm tra Product Versioning
  Tổng product_id       : 2,412
  Tổng (name,size,color): 2,172
  Rows bị lặp identity  : 426  (17.7%)

  Số SKU có >1 version  : 186

  Top 10 SKU nhiều version nhất:
                            n_versions  price_min  price_max  cogs_min  cogs_max
product_name   size color                                                       
VietMode RP-15 XL   blue             3      29.73    4409.37     18.41   3810.71
VietMode RP-16 S    white            3      22.39    4409.37     14.19   4188.90
VietMode RP-17 M    purple           3      19.77    4781.39     11.64   2627.85
VietMode RP-18 L    green            3      19.73    4781.39     11.73   4542.32
VietMode RP-49 M    silver           3    4912.74    6070.52   3428.63   4167.97
VietMode RP-50 L    pink             3    4912.74    6070.52   4573.27   5205.75
VietMode RP-53 M    black            3      22.29    4970.07     14.06   4721.57
VietMode RP-54 L    orange           3      31.49    4970.07  

In [38]:
# ── CELL 2: products — Price & COGS sanity check ────────────
print("\n" + "=" * 65)
print("💰 PRODUCTS — Price / COGS Distribution")
print("=" * 65)

# Kiểm tra ràng buộc cogs < price
violated = prod[prod['cogs'] >= prod['price']]
print(f"  Vi phạm cogs >= price : {len(violated):,} rows")

# Gross margin distribution
prod['gross_margin'] = (prod['price'] - prod['cogs']) / prod['price']
print(f"\n  Gross margin (price−cogs)/price:")
print(prod['gross_margin'].describe().to_string())

# Theo segment
print(f"\n  Gross margin trung bình theo segment:")
print(prod.groupby('segment')['gross_margin'].mean().sort_values(ascending=False).to_string())

# Price distribution — xem scale issue
print(f"\n  Price distribution:")
print(prod['price'].describe().to_string())
print(f"\n  Giá thấp nhất 10 sản phẩm:")
print(prod.nsmallest(10, 'price')[['product_id','product_name','size','color','price','cogs']].to_string(index=False))


💰 PRODUCTS — Price / COGS Distribution
  Vi phạm cogs >= price : 0 rows

  Gross margin (price−cogs)/price:
count   2412.00
mean       0.27
std        0.15
min        0.05
25%        0.10
50%        0.31
75%        0.40
max        0.50

  Gross margin trung bình theo segment:
segment
Standard      0.31
Premium       0.29
All-weather   0.28
Activewear    0.27
Performance   0.26
Balanced      0.26
Trendy        0.24
Everyday      0.24

  Price distribution:
count    2412.00
mean     4928.22
std      4776.74
min         9.06
25%        59.44
50%      4399.60
75%      7720.51
max     40950.00

  Giá thấp nhất 10 sản phẩm:
 product_id      product_name size color  price  cogs
        888 HanoiStreet UR-34    S  pink   9.06  5.18
       1553    VietMode RP-45    M  blue  11.26  6.89
        923 HanoiStreet UE-32   XL  blue  11.61  6.88
       2260  VietMotion RP-57    S   red  12.32  7.19
       1641    VietMode RS-79    M black  13.19  7.46
       1931     UrbanVN RP-06   XL black  13.67  

In [39]:
# ── CELL 3: products — Category / Segment cardinality ───────
print("\n" + "=" * 65)
print("🏷️  PRODUCTS — Category & Segment Structure")
print("=" * 65)

for col in ['category', 'segment']:
    vc = prod[col].value_counts()
    print(f"\n  [{col}]  ({vc.shape[0]} unique values)")
    print(vc.to_string())


🏷️  PRODUCTS — Category & Segment Structure

  [category]  (4 unique values)
category
Streetwear    1320
Outdoor        743
Casual         201
GenZ           148

  [segment]  (8 unique values)
segment
Activewear     598
Everyday       405
Performance    347
Balanced       306
Standard       262
Premium        177
All-weather    169
Trendy         148


In [40]:
# ── CELL 4: geography — Zip/Region sanity ───────────────────
print("\n" + "=" * 65)
print("🗺️  GEOGRAPHY — Scale & Region Check")
print("=" * 65)

geo = dfs['geography'].copy()

print(f"  Tổng zip codes : {geo['zip'].nunique():,}")
print(f"  Tổng cities    : {geo['city'].nunique():,}")
print(f"  Tổng regions   : {geo['region'].nunique():,}")
print(f"  Tổng districts : {geo['district'].nunique():,}")

print(f"\n  Phân phối zip theo region:")
print(geo.groupby('region').agg(
    n_zip=('zip','count'),
    n_city=('city','nunique'),
    n_district=('district','nunique')
).to_string())

# Kiểm tra xem customers.city có khớp geography.city không
cust = dfs['customers'].copy()
cust_merged = cust.merge(geo[['zip','city']].rename(columns={'city':'city_geo'}), on='zip', how='left')
mismatch = cust_merged[cust_merged['city'] != cust_merged['city_geo']]
print(f"\n  customers.city vs geography.city mismatch: {len(mismatch):,} rows ({len(mismatch)/len(cust)*100:.1f}%)")


🗺️  GEOGRAPHY — Scale & Region Check
  Tổng zip codes : 39,948
  Tổng cities    : 42
  Tổng regions   : 3
  Tổng districts : 39

  Phân phối zip theo region:
         n_zip  n_city  n_district
region                            
Central  14512      12          13
East     18929      14          19
West      6507      16           7

  customers.city vs geography.city mismatch: 0 rows (0.0%)


In [41]:
# ── CELL 5: customers — Cohort & Channel analysis ───────────
print("\n" + "=" * 65)
print("👤 CUSTOMERS — Signup Pattern & Demographics")
print("=" * 65)

cust = dfs['customers'].copy()
cust['signup_date'] = pd.to_datetime(cust['signup_date'])
cust['signup_year']  = cust['signup_date'].dt.year

print("  Signup theo năm:")
print(cust['signup_year'].value_counts().sort_index().to_string())

print(f"\n  Gender distribution (incl. null):")
print(cust['gender'].value_counts(dropna=False).to_string())

print(f"\n  Age group distribution (incl. null):")
print(cust['age_group'].value_counts(dropna=False).to_string())

print(f"\n  Acquisition channel:")
print(cust['acquisition_channel'].value_counts(dropna=False).to_string())

# Kiểm tra customers có signup_date SAU ngày đặt đơn đầu tiên của họ
orders = dfs['orders'].copy()
orders['order_date'] = pd.to_datetime(orders['order_date'])
first_order = orders.groupby('customer_id')['order_date'].min().reset_index()
first_order.columns = ['customer_id','first_order_date']

cust_check = cust.merge(first_order, on='customer_id', how='inner')
time_travel = cust_check[cust_check['first_order_date'] < cust_check['signup_date']]
print(f"\n  ⚠️  Customers đặt hàng TRƯỚC ngày signup: {len(time_travel):,} ({len(time_travel)/len(cust_check)*100:.1f}%)")


👤 CUSTOMERS — Signup Pattern & Demographics
  Signup theo năm:
signup_year
2012      957
2013     2989
2014     5034
2015     7133
2016     9202
2017    11078
2018    13011
2019    15058
2020    17211
2021    19154
2022    21103

  Gender distribution (incl. null):
gender
Female        59640
Male          57457
Non-binary     4833

  Age group distribution (incl. null):
age_group
25-34    36342
35-44    31920
45-54    23172
18-24    17039
55+      13457

  Acquisition channel:
acquisition_channel
organic_search    36450
social_media      24448
paid_search       24285
email_campaign    14674
referral          12270
direct             9803

  ⚠️  Customers đặt hàng TRƯỚC ngày signup: 80,623 (89.3%)


In [42]:
# ── CELL 6: promotions — Temporal & Stackable logic ─────────
print("\n" + "=" * 65)
print("🎁 PROMOTIONS — Temporal Validity & Coverage")
print("=" * 65)

promo = dfs['promotions'].copy()
promo['start_date'] = pd.to_datetime(promo['start_date'])
promo['end_date']   = pd.to_datetime(promo['end_date'])
promo['duration_days'] = (promo['end_date'] - promo['start_date']).dt.days

# Promos với end < start (invalid)
invalid_date = promo[promo['end_date'] < promo['start_date']]
print(f"  Promos với end_date < start_date: {len(invalid_date):,}")

# Overlap check — promos cùng applicable_category có bị overlap không?
print(f"\n  Stackable flag distribution:")
print(promo['stackable_flag'].value_counts().to_string())

print(f"\n  Duration stats (ngày):")
print(promo['duration_days'].describe().to_string())

print(f"\n  Promo type:")
print(promo['promo_type'].value_counts().to_string())

print(f"\n  applicable_category (null = all):")
print(promo['applicable_category'].value_counts(dropna=False).to_string())

# Cross-check: promo_ids trong order_items có tồn tại trong promotions không?
oi = dfs['order_items'].copy()
all_promo_in_oi = set(oi['promo_id'].dropna().unique()) | set(oi['promo_id_2'].dropna().unique())
all_promo_in_master = set(promo['promo_id'].unique())
orphan_promos = all_promo_in_oi - all_promo_in_master
print(f"\n  ⚠️  promo_id trong order_items không có trong promotions master: {len(orphan_promos):,}")
if orphan_promos:
    print(f"     Samples: {list(orphan_promos)[:5]}")


🎁 PROMOTIONS — Temporal Validity & Coverage
  Promos với end_date < start_date: 0

  Stackable flag distribution:
stackable_flag
0    38
1    12

  Duration stats (ngày):
count   50.00
mean    33.54
std      5.81
min     29.00
25%     30.00
50%     30.50
75%     34.00
max     45.00

  Promo type:
promo_type
percentage    45
fixed          5

  applicable_category (null = all):
applicable_category
NaN           40
Streetwear     5
Outdoor        5

  ⚠️  promo_id trong order_items không có trong promotions master: 0


In [1]:
# ── RETENTION DEEP DIVE ──────────────────────────────────────
# Dùng first_order_year thay signup_year cho cohort

import pandas as pd
import numpy as np
from pathlib import Path

INPUT_DIR = Path('input')
orders      = pd.read_csv(INPUT_DIR / 'orders.csv', parse_dates=['order_date'])
order_items = pd.read_csv(INPUT_DIR / 'order_items.csv')
customers   = pd.read_csv(INPUT_DIR / 'customers.csv')

orders_d = orders[orders['order_status'] == 'delivered'].copy()
orders_d['year'] = orders_d['order_date'].dt.year

order_rev = (order_items.assign(rev=lambda x: x['unit_price']*x['quantity'] - x['discount_amount'])
             .groupby('order_id')['rev'].sum().reset_index()
             .rename(columns={'rev':'order_revenue'}))

txn = orders_d.merge(order_rev, on='order_id', how='left')
txn = txn.merge(customers[['customer_id','age_group','gender']], on='customer_id', how='left')

# Customer summary dùng first_order thay signup
cust = txn.groupby('customer_id').agg(
    n_orders        = ('order_id', 'count'),
    total_revenue   = ('order_revenue', 'sum'),
    avg_order_value = ('order_revenue', 'mean'),
    first_order     = ('order_date', 'min'),
    last_order      = ('order_date', 'max'),
    age_group       = ('age_group', 'first'),
    gender          = ('gender', 'first'),
).reset_index()

cust['first_order_year'] = cust['first_order'].dt.year
cust['last_order_year']  = cust['last_order'].dt.year
cust['lifetime_days']    = (cust['last_order'] - cust['first_order']).dt.days

# ── LUẬN ĐIỂM 1: Cohort retention dùng first_order_year ──────
print("=" * 60)
print("LUẬN ĐIỂM 1 — Cohort Retention (first_order_year)")
print("=" * 60)

# Tạo bảng: cohort x active_year → % còn mua
cohort_activity = txn.groupby(['customer_id','year'])['order_id'].count().reset_index()
cohort_activity = cohort_activity.merge(
    cust[['customer_id','first_order_year']], on='customer_id')
cohort_activity['years_since_first'] = cohort_activity['year'] - cohort_activity['first_order_year']

# % customers trong mỗi cohort còn active sau N năm
cohort_size = cust.groupby('first_order_year')['customer_id'].count()
retention_matrix = (cohort_activity
    .groupby(['first_order_year','years_since_first'])['customer_id']
    .nunique()
    .unstack(fill_value=0))
retention_pct = retention_matrix.div(cohort_size, axis=0) * 100

print("\nRetention rate (%) by cohort — Year 0 = năm đầu mua:")
print(retention_pct.round(1).to_string())

# Retention năm đầu tiên (Year 1) theo cohort — đây là số quan trọng nhất
print("\nYear-1 retention (% quay lại mua năm tiếp theo):")
if 1 in retention_pct.columns:
    print(retention_pct[1].sort_index().round(1).to_string())

C:\Users\Dell Gaming\AppData\Local\Temp\ipykernel_18652\2955258935.py:10: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv(INPUT_DIR / 'order_items.csv')


LUẬN ĐIỂM 1 — Cohort Retention (first_order_year)

Retention rate (%) by cohort — Year 0 = năm đầu mua:
years_since_first     0     1     2     3     4     5     6     7     8     9     10
first_order_year                                                                    
2012               100.0  60.2  60.3  61.3  61.4  59.3  57.1  42.6  38.3  37.3  37.7
2013               100.0  48.1  48.4  48.3  46.8  44.9  32.4  28.8  27.6  28.1   0.0
2014               100.0  33.9  34.9  33.4  31.4  21.3  19.2  17.8  18.9   0.0   0.0
2015               100.0  27.5  26.7  24.5  16.1  14.1  13.2  13.8   0.0   0.0   0.0
2016               100.0  21.2  19.7  12.7  10.8  10.5  10.8   0.0   0.0   0.0   0.0
2017               100.0  16.1  11.2   9.3   9.3   8.2   0.0   0.0   0.0   0.0   0.0
2018               100.0   9.6   8.0   7.6   8.8   0.0   0.0   0.0   0.0   0.0   0.0
2019               100.0   8.0   7.4   7.4   0.0   0.0   0.0   0.0   0.0   0.0   0.0
2020               100.0   6.5   5.9   0.0   0

In [2]:
# ── LUẬN ĐIỂM 2: High-value vs Low-value churn asymmetry ─────
print("\n" + "=" * 60)
print("LUẬN ĐIỂM 2 — Ai đang churn: High-value hay Low-value?")
print("=" * 60)

# Chia customers thành 3 tier bằng CLV
cust['clv_tier'] = pd.qcut(cust['total_revenue'],
                            q=[0, 0.33, 0.67, 1.0],
                            labels=['Low', 'Mid', 'High'])

# Churned = last_order trước 2022
CHURN_CUTOFF = pd.Timestamp('2022-01-01')
cust['is_churned'] = cust['last_order'] < CHURN_CUTOFF

print("\nChurn rate by CLV tier:")
churn_by_tier = cust.groupby('clv_tier').agg(
    n_customers    = ('customer_id', 'count'),
    churn_rate     = ('is_churned', 'mean'),
    avg_clv        = ('total_revenue', 'mean'),
    avg_orders     = ('n_orders', 'mean'),
    avg_lifetime   = ('lifetime_days', 'mean')
).round(2)
print(churn_by_tier.to_string())

# Quan trọng hơn: trong số High-value churners, họ churn vào năm nào?
high_churners = cust[(cust['clv_tier'] == 'High') & (cust['is_churned'])]
print(f"\nHigh-value churners: {len(high_churners):,}")
print("Năm cuối mua của High-value churners:")
print(high_churners['last_order_year'].value_counts().sort_index().to_string())


LUẬN ĐIỂM 2 — Ai đang churn: High-value hay Low-value?

Churn rate by CLV tier:
          n_customers  churn_rate    avg_clv  avg_orders  avg_lifetime
clv_tier                                                              
Low             28088        0.93   18580.02        1.47        411.33
Mid             28939        0.82   84534.47        3.62       1609.38
High            28088        0.52  340001.34       13.20       2926.00

High-value churners: 14,642
Năm cuối mua của High-value churners:
last_order_year
2012       2
2013       1
2014      15
2015      66
2016     199
2017     578
2018    1596
2019    2171
2020    3585
2021    6429


C:\Users\Dell Gaming\AppData\Local\Temp\ipykernel_18652\3774310876.py:16: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  churn_by_tier = cust.groupby('clv_tier').agg(


In [3]:
# ── LUẬN ĐIỂM 3: Inter-purchase gap trend theo cohort ─────────
print("\n" + "=" * 60)
print("LUẬN ĐIỂM 3 — Gap giữa lần mua 1 và 2 có tăng không?")
print("=" * 60)

# Tính gap lần mua 1 → 2 per customer
txn_sorted = txn.sort_values(['customer_id', 'order_date'])
txn_sorted['prev_order_date'] = txn_sorted.groupby('customer_id')['order_date'].shift(1)
txn_sorted['gap_days'] = (txn_sorted['order_date'] - txn_sorted['prev_order_date']).dt.days
txn_sorted['purchase_rank'] = txn_sorted.groupby('customer_id').cumcount() + 1

# Chỉ lấy gap từ lần 1 → 2
gap_1to2 = txn_sorted[txn_sorted['purchase_rank'] == 2][['customer_id','order_date','gap_days']].copy()
gap_1to2['year_of_second_purchase'] = gap_1to2['order_date'].dt.year

print("\nMedian gap 1→2 theo năm của lần mua thứ 2:")
print(gap_1to2.groupby('year_of_second_purchase')['gap_days'].median().round(0).to_string())

# Kết hợp với cohort: gap theo first_order_year
gap_with_cohort = gap_1to2.merge(cust[['customer_id','first_order_year']], on='customer_id')
print("\nMedian gap 1→2 theo cohort (first_order_year):")
print(gap_with_cohort.groupby('first_order_year')['gap_days'].median().round(0).to_string())


LUẬN ĐIỂM 3 — Gap giữa lần mua 1 và 2 có tăng không?

Median gap 1→2 theo năm của lần mua thứ 2:
year_of_second_purchase
2012      40.0
2013     144.0
2014     323.0
2015     501.0
2016     673.0
2017     807.0
2018    1074.0
2019    1259.0
2020    1555.0
2021    1826.0
2022    2150.0

Median gap 1→2 theo cohort (first_order_year):
first_order_year
2012    238.0
2013    314.0
2014    422.0
2015    453.0
2016    454.0
2017    426.0
2018    488.0
2019    456.0
2020    321.0
2021    211.0
2022     64.0


In [4]:
# ── LUẬN ĐIỂM 4: Revenue concentration risk ──────────────────
print("\n" + "=" * 60)
print("LUẬN ĐIỂM 4 — Pareto & Revenue Concentration Risk")
print("=" * 60)

cust_sorted = cust.sort_values('total_revenue', ascending=False).reset_index(drop=True)
cust_sorted['cum_rev_pct'] = cust_sorted['total_revenue'].cumsum() / cust_sorted['total_revenue'].sum() * 100
cust_sorted['cum_cust_pct'] = (cust_sorted.index + 1) / len(cust_sorted) * 100

for pct in [10, 20, 30]:
    rev = cust_sorted[cust_sorted['cum_cust_pct'] <= pct]['cum_rev_pct'].max()
    n   = int(len(cust_sorted) * pct / 100)
    print(f"Top {pct}% KH ({n:,} người) → {rev:.1f}% Revenue")

# Trong top 20%, churn rate là bao nhiêu?
top20_cutoff = cust_sorted.iloc[int(len(cust_sorted)*0.2)]['total_revenue']
cust['is_top20'] = cust['total_revenue'] >= top20_cutoff
print(f"\nChurn rate của top 20% KH: {cust[cust['is_top20']]['is_churned'].mean()*100:.1f}%")
print(f"Churn rate của bottom 80% KH: {cust[~cust['is_top20']]['is_churned'].mean()*100:.1f}%")

# Nếu top 20% churn rate cao → đây là rủi ro tập trung cực kỳ nguy hiểm
# Revenue at risk = % Rev của top 20% × churn rate của họ
rev_at_risk_pct = cust[cust['is_top20']]['total_revenue'].sum() / cust['total_revenue'].sum() * 100
top20_churn = cust[cust['is_top20']]['is_churned'].mean()
print(f"\nRevenue at risk (top 20% × churn rate): {rev_at_risk_pct * top20_churn:.1f}% total revenue")


LUẬN ĐIỂM 4 — Pareto & Revenue Concentration Risk
Top 10% KH (8,511 người) → 39.1% Revenue
Top 20% KH (17,023 người) → 59.5% Revenue
Top 30% KH (25,534 người) → 73.1% Revenue

Churn rate của top 20% KH: 43.3%
Churn rate của bottom 80% KH: 84.2%

Revenue at risk (top 20% × churn rate): 25.8% total revenue
